<a href="https://colab.research.google.com/github/GehadHelmy/github_SNA/blob/main/Project_SNA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Libraries**

In [41]:

import pandas as pd
import numpy as np
import json
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import networkx.algorithms.community as nx_comm
from matplotlib.collections import LineCollection
from sklearn.metrics import normalized_mutual_info_score
from scipy.sparse import csr_matrix
import graphistry

## **Dataset Understanding**

In [42]:
edges_df = pd.read_csv("musae_git_edges.csv")
target_df = pd.read_csv("musae_git_target.csv")
with open("musae_git_features.json", "r") as file:
    features_data = json.load(file)

In [43]:
print("Edges Shape:", edges_df.shape)
print("Target Shape:", target_df.shape)
print("Number of Feature Records:", len(features_data))

Edges Shape: (289003, 2)
Target Shape: (37700, 3)
Number of Feature Records: 37700


In [44]:
print("Edges:")
display(edges_df.head())
print("Target:")
display(target_df.head())

Edges:


,id_1,id_2
0,0,23977
1,1,34526
2,1,2370
3,1,14683
4,1,29982


Target:


,id,name,ml_target
0,0,Eiryyy,0
1,1,shawflying,0
2,2,JpMCarrilho,1
3,3,SuhwanCha,0
4,4,sunilangadi2,1


In [45]:
print(edges_df.columns)
print(target_df.columns)

Index(['id_1', 'id_2'], dtype='object')
Index(['id', 'name', 'ml_target'], dtype='object')


In [46]:
for i, (node_id, features) in enumerate(features_data.items()):
    print(node_id, features)
    if i == 4:
        break

0 [1574, 3773, 3571, 2672, 2478, 2534, 3129, 3077, 1171, 2045, 1539, 902, 1532, 2472, 1122, 2480, 3098, 2115, 1578]
1 [1193, 376, 73, 290, 3129, 1852, 3077, 1171, 1022, 2045, 536, 2040, 1533, 1532, 2472, 673, 798]
2 [1574, 3773, 925, 1728, 2815, 2963, 3077, 364, 1171, 536, 1867, 2472, 1122, 2532, 664, 28, 3311, 1768, 869]
3 [3964, 3773, 4003, 928, 1852, 3077, 364, 1022, 3763, 2045, 3859, 3771, 234, 664, 703]
4 [1929, 3773, 1793, 3511, 1290, 3129, 3077, 364, 1171, 1022, 2045, 536, 1119, 91, 1532, 3771, 1122, 703, 1001]


## **Data Preprocessing & Cleaning**

In [47]:
print("Edges Missing Values:")
print(edges_df.isnull().sum())
print("\nTarget Missing Values:")
print(target_df.isnull().sum())

Edges Missing Values:
id_1    0
id_2    0
dtype: int64

Target Missing Values:
id           0
name         0
ml_target    0
dtype: int64


In [48]:
edges_df["source"] = edges_df["id_1"].astype(int)

edges_df["target"] = edges_df["id_2"].astype(int)

edges_df["source_id"] = edges_df[["source", "target"]].min(axis=1)

edges_df["target_id"] = edges_df[["source", "target"]].max(axis=1)

print("Original Edges:", len(edges_df))
edges_df = edges_df.drop_duplicates(
    subset=["source_id", "target_id"])
edges_df = edges_df[["source_id", "target_id"]]
print("After Removing Duplicates:", len(edges_df))

Original Edges: 289003
After Removing Duplicates: 289003


In [49]:
self_loops = edges_df[
    edges_df["source_id"] == edges_df["target_id"]]
print("Number of Self Loops:", len(self_loops))

Number of Self Loops: 0


In [50]:
print("Duplicate Developer IDs:",target_df["id"].duplicated().sum())

Duplicate Developer IDs: 0


## **Graph Construction**

In [51]:
feature_nodes = set(map(int, features_data.keys()))
len(feature_nodes)

37700

In [52]:
G = nx.Graph()
G.add_nodes_from(feature_nodes)
G.add_edges_from(
    edges_df[["source_id", "target_id"]].itertuples(
        index=False,
        name=None
    ))
print("Number of Nodes:", G.number_of_nodes())
print("Number of Edges:", G.number_of_edges())

Number of Nodes: 37700
Number of Edges: 289003


### **Edge List**

In [53]:
edge_list = list(G.edges())
print(edge_list[:10])

[(0, 23977), (1, 34526), (1, 2370), (1, 14683), (1, 29982), (1, 21142), (1, 20363), (1, 23830), (1, 34035), (2, 3812)]


In [54]:
edge_list_df = pd.DataFrame(
    edge_list,
    columns=["Source", "Target"])
edge_list_df.to_csv("edge_list.csv", index=False)
edge_list_df.head()

,Source,Target
0,0,23977
1,1,34526
2,1,2370
3,1,14683
4,1,29982


### **Adjacency List**

In [55]:
adjacency_list = nx.to_dict_of_lists(G)
for node, neighbors in list(adjacency_list.items())[:5]:
    print(node, ":", neighbors)

0 : [23977]
1 : [34526, 2370, 14683, 29982, 21142, 20363, 23830, 34035]
2 : [3812]
3 : [4950, 18029, 3358, 34935, 5916]
4 : [2865, 9342]


In [56]:
with open("adjacency_list.json", "w") as file:
    json.dump(adjacency_list, file)

### **Adjacency Matrix**

In [57]:
A = nx.to_scipy_sparse_array(
    G,
    format="csr",
    dtype=np.int8)
print("Matrix Shape:", A.shape)
print("Number of Connections:", A.nnz)
print("Matrix Type:", type(A))

Matrix Shape: (37700, 37700)
Number of Connections: 578006
Matrix Type: <class 'scipy.sparse._csr.csr_array'>


In [58]:
graphistry.register(api=3, username="", password="")
g = graphistry.edges(
    edges_df,
    source="source_id",
    destination="target_id"
)

g2 = g.scene_settings(
    show_arrows=False,
    point_size=2.0,
    edge_curvature=0.0,
    point_opacity=1.0,
    edge_opacity=0.47
)

g2.plot()

ERROR:graphistry.utils.requests:HTTP 400 error - response content (JSON): {
  "password": [
    "This field may not be blank."
  ],
  "username": [
    "This field may not be blank."
  ]
}
ERROR:graphistry.arrow_uploader:Error: <Response [400]>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/graphistry/arrow_uploader.py", line 331, in _finalize_login
    raise Exception(f"Login failed with HTTP {out.status_code}: {out.text}")
Exception: Login failed with HTTP 400: {"password":["This field may not be blank."],"username":["This field may not be blank."]}


Exception: Login failed with HTTP 400: {"password":["This field may not be blank."],"username":["This field may not be blank."]}

In [ ]:
graph_url = g2.plot(render=False)
print(graph_url)

# **Centrality Measures**

In [ ]:
degree = dict(G.degree())
degree

In [ ]:
sorted_degree = sorted(degree.items(),key=lambda x: x[1],)

print(sorted_degree)

In [ ]:
pagerank = nx.pagerank(G)
pagerank

In [ ]:
eigenvector_centrality = nx.eigenvector_centrality_numpy(G)

In [ ]:
sorted_eigenvector = sorted(eigenvector_centrality.items(), key=lambda item: item[1])
sorted_eigenvector[:10]

In [ ]:
centrality_df = pd.DataFrame({
    'Node': list(G.nodes()),
    'Degree': [degree[n] for n in G.nodes()],
    'PageRank': [pagerank[n] for n in G.nodes()],
    'Eigenvector': [eigenvector_centrality[n] for n in G.nodes()]
})
centrality_df

In [ ]:
degree_threshold = centrality_df['Degree'].quantile(0.75)
pagerank_threshold = centrality_df['PageRank'].quantile(0.75)
eigenvector_threshold = centrality_df['Eigenvector'].quantile(0.75)

In [ ]:
filtered_nodes = centrality_df[
    (centrality_df['Degree'] >= degree_threshold) &
    (centrality_df['PageRank'] >= pagerank_threshold) &
    (centrality_df['Eigenvector'] >= eigenvector_threshold)
]['Node'].tolist()

In [ ]:
G_filtered = G.subgraph(filtered_nodes).copy()

In [ ]:
# betweenness = nx.betweenness_centrality(G_filtered)

In [ ]:
betweenness

In [ ]:
closeness = nx.closeness_centrality(G_filtered)
closeness

In [ ]:
results_df = pd.DataFrame({
    'Node': list(G_filtered.nodes()),
    'Degree': [degree[n] for n in G_filtered.nodes()],
    'PageRank': [pagerank[n] for n in G_filtered.nodes()],
    'Eigenvector': [eigenvector_centrality[n] for n in G_filtered.nodes()],
    'Betweenness': [betweenness[n] for n in G_filtered.nodes()],
    'Closeness': [closeness[n] for n in G_filtered.nodes()]
})

In [ ]:
results_df

In [ ]:
degrees = [d for _, d in G_filtered.degree()]
deg_counts = Counter(degrees)
deg_df = (pd.DataFrame(sorted(deg_counts.items()), columns=["Degree", "Count"])
            .assign(P_k=lambda d: d["Count"] / G_filtered.number_of_nodes()))
print(deg_df.head(10))
print(f"Mean: {np.mean(degrees):.2f} | Median: {np.median(degrees)} | Max: {max(degrees)}")


In [ ]:
top = deg_df.head(50)

plt.figure(figsize=(20, 5))
plt.bar(top["Degree"], top["P_k"], color="#ff6161")
plt.xticks(top["Degree"])
plt.xlabel("Degree (k)")
plt.ylabel("P(k)")
plt.title("Degree Distribution (first 50 degrees)")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.ravel(), cols):
    t = results_df.nlargest(10, col)
    ax.barh(t["Node"].astype(str), t[col], color="steelblue")
    ax.invert_yaxis(); ax.set_title(f"Top 10 by {col}")
axes.ravel()[-1].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
results_df[cols[1:]].plot(kind="box", subplots=True, layout=(1, 4), figsize=(14, 4))
plt.tight_layout(); plt.show()

# **Community Detection & Modularity**

In [ ]:
communities = nx_comm.louvain_communities(G, seed=42)
print("Number of communities:", len(communities))
print("Top 5 sizes:", sorted(map(len, communities), reverse=True)[:5])

node_comm = {n: i for i, c in enumerate(communities) for n in c}
centrality_df["Community"] = centrality_df["Node"].map(node_comm)

In [ ]:
lab = target_df.set_index("id")["ml_target"]
nodes = [n for n in G.nodes() if n in lab.index]
nmi = normalized_mutual_info_score([lab[n] for n in nodes], [node_comm[n] for n in nodes])
print(f"NMI vs ml_target: {nmi:.3f}")

In [ ]:
rows = []
for i, c in enumerate(communities):
    top = max(c, key=lambda n: degree[n])
    rows.append({"Community": i, "Size": len(c),
                 "AvgDegree": np.mean([degree[n] for n in c]),
                 "TopNode": top, "TopNodeDegree": degree[top]})
comm_df = pd.DataFrame(rows).sort_values("Size", ascending=False)
comm_df.head(10)

In [ ]:
sizes = sorted(map(len, communities), reverse=True)
plt.figure(figsize=(8, 4.5))
plt.bar(range(1, 11), sizes[:10], color="#ff6161")
plt.xlabel("Community rank"); plt.ylabel("Nodes"); plt.title("Top 10 Communities by Size")
plt.show()

In [ ]:
Q = nx_comm.modularity(G, communities)
print(f"Modularity: {Q:.4f}")

In [ ]:

TOP_COMMS, PER_COMM = 5, 300

top_comms = sorted(communities, key=len, reverse=True)[:TOP_COMMS]
keep = [n for c in top_comms
        for n in sorted(c, key=lambda x: degree[x], reverse=True)[:PER_COMM]]
H = G.subgraph(keep)

pos = nx.spring_layout(H, seed=42, iterations=30, k=1 / np.sqrt(len(H)))

edges = np.array([[pos[u], pos[v]] for u, v in H.edges()])
fig, ax = plt.subplots(figsize=(10, 8))
ax.add_collection(LineCollection(edges, colors="lightgray", linewidths=0.3, alpha=0.5))
xy = np.array([pos[n] for n in H.nodes()])
ax.scatter(xy[:, 0], xy[:, 1], s=[degree[n] * 0.5 + 5 for n in H.nodes()],
           c=[node_comm[n] for n in H.nodes()], cmap="tab10", linewidths=0)
ax.autoscale(); ax.axis("off")
plt.show()

In [ ]:
# Edges inside the same community only
intra = [(u, v) for u, v in H.edges() if node_comm[u] == node_comm[v]]
rng = np.random.default_rng(0)
comm_ids = sorted({node_comm[n] for n in H.nodes()})
centers = {c: np.array([np.cos(2*np.pi*i/len(comm_ids)),
                        np.sin(2*np.pi*i/len(comm_ids))])
           for i, c in enumerate(comm_ids)}
init = {n: centers[node_comm[n]] + rng.normal(0, 0.1, 2) for n in H.nodes()}

pos = nx.spring_layout(H, pos=init, iterations=20, k=0.15, seed=42)

edges = np.array([[pos[u], pos[v]] for u, v in intra])
fig, ax = plt.subplots(figsize=(10, 8))
ax.add_collection(LineCollection(edges, colors="lightgray", linewidths=0.2, alpha=0.3))
xy = np.array([pos[n] for n in H.nodes()])
ax.scatter(xy[:, 0], xy[:, 1], s=[degree[n] * 0.5 + 5 for n in H.nodes()],
           c=[node_comm[n] for n in H.nodes()], cmap="tab10", linewidths=0)
ax.autoscale(); ax.axis("off"); plt.show()